In [9]:

import os
import zipfile
import pandas as pd
import numpy as np
import torch
import catboost as cb
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm # Для красивого отображения прогресса

# Проверим доступность GPU, это значительно ускорит создание эмбеддингов
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

Используемое устройство: cpu


In [ ]:
# --- Распаковка архивов ---
print("=== Распаковка ZIP-архивов ===\n")
files_to_unzip = ['train.tsv.zip', 'test.tsv.zip', 'reviews.txv.zip']
for zip_filename in files_to_unzip:
    if os.path.exists(zip_filename):
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"УСПЕХ: Архив '{zip_filename}' распакован.")
    else:
        print(f"ПРЕДУПРЕЖДЕНИЕ: Архив '{zip_filename}' не найден.")

# --- Загрузка и объединение данных ---
print("\n=== Загрузка и объединение данных ===\n")
try:
    train_data = pd.read_csv('train.tsv', sep='\t')
    test_data = pd.read_csv('test.tsv', sep='\t')
    reviews_data = pd.read_csv('reviews.tsv', sep='\t')

    reviews_aggregated = reviews_data.groupby('id')['text'].apply(lambda x: ' '.join(x)).reset_index()
    reviews_aggregated.rename(columns={'text': 'full_review_text'}, inplace=True)

    train_full = pd.merge(train_data, reviews_aggregated, on='id', how='left')
    test_full = pd.merge(test_data, reviews_aggregated, on='id', how='left')

    train_full['full_review_text'] = train_full['full_review_text'].fillna('')
    test_full['full_review_text'] = test_full['full_review_text'].fillna('')
    print("Данные успешно загружены и объединены.")
    display(train_full.head(2))

except Exception as e:
    print(f"Произошла ошибка: {e}")

=== Распаковка ZIP-архивов ===

УСПЕХ: Архив 'train.tsv.zip' распакован.
УСПЕХ: Архив 'test.tsv.zip' распакован.
УСПЕХ: Архив 'reviews.txv.zip' распакован.

=== Загрузка и объединение данных ===



In [ ]:
import re
from nltk.corpus import stopwords
from pymorphy3 import MorphAnalyzer
from tqdm.notebook import tqdm

# --- Инициализируем инструменты один раз, чтобы не делать это для каждого текста ---
# Это важно для производительности!
morph = MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))

def preprocess_text(text):
    """
    Функция для лемматизации, удаления стоп-слов и очистки текста.
    """
    # 1. Проверка на случай, если текст пустой или не является строкой
    if not isinstance(text, str) or len(text) == 0:
        return ""

    # 2. Приводим к нижнему регистру и удаляем все, кроме букв и пробелов
    text = text.lower()
    text = re.sub(r'[^а-яё\s]', '', text)

    # 3. Разбиваем текст на слова (токенизация)
    words = text.split()

    # 4. Лемматизация и удаление стоп-слов в одном цикле
    clean_words = []
    for word in words:
        # Пропускаем стоп-слова
        if word not in russian_stopwords:
            # Приводим слово к нормальной форме
            normal_form = morph.parse(word)[0].normal_form
            clean_words.append(normal_form)

    # 5. Соединяем очищенные слова обратно в строку
    return " ".join(clean_words)
tqdm.pandas()

print("Начинаю предобработку текстов в обучающем наборе...")
# Создаем новую колонку с очищенными отзывами
# progress_apply вместо apply покажет прогресс
train_full['clean_review'] = train_full['full_review_text'].progress_apply(preprocess_text)
print("Обработка обучающего набора завершена.")

print("\nНачинаю предобработку текстов в тестовом наборе...")
test_full['clean_review'] = test_full['full_review_text'].progress_apply(preprocess_text)
print("Обработка тестового набора завершена.")

# --- Посмотрим на результат ---
print("\nПример предобработки:")
display(train_full[['full_review_text', 'clean_review']].head())


In [ ]:
display(test_full[['clean_review']].head())


In [ ]:
!pip install pymorphy3

In [ ]:


for df in [train_full, test_full]:
    df['target'] = math.log1p(df['target'])
    df['review_len'] = df['full_review_text'].str.len() # Длина отзыва в символах
    df['review_words'] = df['full_review_text'].str.split().str.len() # Количество слов
    df['review_sentences'] = df['full_review_text'].str.count('\.') # Примерное кол-во

if 'full_review_text' in train_full.columns:
    train_full = train_full.drop('full_review_text', axis=1)
if 'full_review_text' in test_full.columns:
    test_full = test_full.drop('full_review_text', axis=1)
COLS_TO_DROP = ['id', 'name', 'address', 'coordinates', 'target']

FEATURES_COLS = [col for col in train_full.columns if col not in COLS_TO_DROP]

# Явно указываем, какие из них категориальные, а какие текстовые
CATEGORICAL_COLS = ['category']
TEXT_COLS = ['clean_review']

# Все остальные будут числовыми (CatBoost определит это сам)

y_train = train_full['target']
X_train = train_full[FEATURES_COLS]
X_test = test_full[FEATURES_COLS]

# Находим все признаки, кроме удаляемых и целевойНY

print(f"Размер обучающей выборки (X_train): {X_train.shape}")
print(f"Используемые признаки: {len(FEATURES_COLS)} колонок")

In [ ]:
import catboost as cb
from sklearn.model_selection import train_test_split

# Разделим данные для валидации, чтобы отслеживать качество
X_train_part, X_val, y_train_part, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# --- Настройка и обучение модели CatBoost ---
model_cb_native = cb.CatBoostRegressor(
    iterations=3000,
    learning_rate=0.03,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=42,
    verbose=100,

    # --- КЛЮЧЕВЫЕ ПАРАМЕТРЫ ---
    cat_features=CATEGORICAL_COLS,  # Указываем категориальные признаки
    text_features=TEXT_COLS,       # Указываем текстовые признаки!

    task_type="GPU" if torch.cuda.is_available() else "CPU",
    early_stopping_rounds=300
)

print("\n--- Запуск обучения CatBoost с встроенной обработкой текста ---")
model_cb_native.fit(
    X_train_part, y_train_part,
    eval_set=(X_val, y_val),
    use_best_model=True
)
print("--- Обучение завершено ---")

In [ ]:
print("\n--- Получение предсказаний для тестового набора ---")
predictions_native = math.expm1(model_cb_native.predict(X_test))

# --- Формирование файла submission.csv ---
submission_df_native = pd.DataFrame({
    'id': test_full['id'],
    'target': predictions_native
})

submission_df_native['target'] = submission_df_native['target'].clip(1, 5)
submission_df_native.to_csv('submission_catboost_native_text.csv', index=False)

print("\nФайл 'submission_catboost_native_text.csv' успешно создан.")
display(submission_df_native.head())

# Task
Implement text preprocessing using spaCy for the 'review' column in the `train` and `test` dataframes.

## Install spacy and the russian language model

### Subtask:
Add a cell to install the necessary libraries and models.


**Reasoning**:
The first step is to install the necessary libraries and models for text preprocessing with spaCy.



In [ ]:
!pip install spacy
!python -m spacy download ru_core_news_sm --quiet

## Load the spacy model

### Subtask:
Load the installed Russian language model.


**Reasoning**:
Import the spacy library and load the Russian language model.



In [ ]:
import spacy

nlp = spacy.load("ru_core_news_sm")
print("Russian language model loaded successfully.")

## Rewrite the preprocess text function

### Subtask:
Modify the existing `preprocess_text` function to use spaCy for tokenization and lemmatization, and update it to use the loaded spaCy model `nlp`.


**Reasoning**:
Modify the preprocess_text function to use the loaded spaCy model for tokenization and lemmatization.



In [ ]:
import re
from nltk.corpus import stopwords
import spacy
from tqdm.notebook import tqdm

# Use the loaded spaCy model
nlp = spacy.load("ru_core_news_sm")
russian_stopwords = set(stopwords.words('russian'))

def preprocess_text(text):
    """
    Функция для лемматизации, удаления стоп-слов и очистки текста с использованием spaCy.
    """
    if not isinstance(text, str) or len(text) == 0:
        return ""

    # Process text using spaCy
    doc = nlp(text.lower()) # Process lowercase text

    clean_tokens = []
    for token in doc:
        # Check if token is alphabetic and not a stop word
        if token.is_alpha and token.text not in russian_stopwords:
            clean_tokens.append(token.lemma_)

    return " ".join(clean_tokens)

tqdm.pandas()

print("Начинаю предобработку текстов в обучающем наборе...")
train_full['clean_review'] = train_full['full_review_text'].progress_apply(preprocess_text)
print("Обработка обучающего набора завершена.")

print("\nНачинаю предобработку текстов в тестовом наборе...")
test_full['clean_review'] = test_full['full_review_text'].progress_apply(preprocess_text)
print("Обработка тестового набора завершена.")

print("\nПример предобработки:")
display(train_full[['full_review_text', 'clean_review']].head())

**Reasoning**:
The previous command failed because the NLTK stopwords were not downloaded. Download the stopwords using NLTK.



In [ ]:
import nltk
nltk.download('stopwords')

**Reasoning**:
Now that the NLTK stopwords are downloaded, retry the previous code block to define and use the updated preprocess_text function.



In [ ]:
import re
from nltk.corpus import stopwords
import spacy
from tqdm.notebook import tqdm

# Use the loaded spaCy model
nlp = spacy.load("ru_core_news_sm")
russian_stopwords = set(stopwords.words('russian'))

def preprocess_text(text):
    """
    Функция для лемматизации, удаления стоп-слов и очистки текста с использованием spaCy.
    """
    if not isinstance(text, str) or len(text) == 0:
        return ""

    # Process text using spaCy
    doc = nlp(text.lower()) # Process lowercase text

    clean_tokens = []
    for token in doc:
        # Check if token is alphabetic and not a stop word
        if token.is_alpha and token.text not in russian_stopwords:
            clean_tokens.append(token.lemma_)

    return " ".join(clean_tokens)

tqdm.pandas()

print("Начинаю предобработку текстов в обучающем наборе...")
train_full['clean_review'] = train_full['full_review_text'].progress_apply(preprocess_text)
print("Обработка обучающего набора завершена.")

print("\nНачинаю предобработку текстов в тестовом наборе...")
test_full['clean_review'] = test_full['full_review_text'].progress_apply(preprocess_text)
print("Обработка тестового набора завершена.")

print("\nПример предобработки:")
display(train_full[['full_review_text', 'clean_review']].head())